# Sesgo en Modelos de Lenguaje (LLMs)
## Una perspectiva estadística: del Cap. 6 al mundo real

**Curso de Posgrado en Estadística – Universidad Nacional del Sur**

---

### Hilo conductor

En el Capítulo 6 definimos sesgo de un estimador como:
$$b(\hat{\theta}) = E(\hat{\theta}) - \theta$$

Un LLM puede pensarse como una **familia de estimadores** $\hat{f}_\theta$ entrenados para aproximar una distribución desconocida $P^*(x)$ sobre texto. El "sesgo" toma múltiples formas según la capa que analicemos:

| Capa | Analogía con Cap. 6 | Tipo de sesgo |
|------|--------------------|--------------|
| Datos de entrenamiento | Muestra no representativa | Sesgo estadístico (de selección) |
| Modelo como estimador | $\hat{\theta}$ vs $\theta$ | Sesgo en estimación / double descent |
| Distribución cultural/geográfica | $E[\hat{\theta}] \neq \theta$ para subpoblaciones | Sesgo sistemático diferencial |
| Outputs y fairness | ECM diferencial entre grupos | Sesgo en consecuencias |

Este notebook recorre los **cuatro ángulos** con demostraciones numéricas.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from scipy import stats
from collections import Counter

rng = np.random.default_rng(42)
plt.rcParams.update({
    'figure.dpi': 110,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'font.size': 11
})

---
## Parte 1 – Sesgo estadístico en el entrenamiento

### 1.1 El problema: los datos de entrenamiento no son una m.a. de $P^*$

Un LLM aprende a modelar $P^*(x)$, la distribución de todo el lenguaje humano. Pero el corpus de entrenamiento es una **muestra sesgada**: internet sobrerepresenta inglés, países de altos ingresos, y texto producido después de ~1990.

**Analogía formal.** Sea $P^*$ la distribución verdadera y $P_{train}$ la del corpus:

$$P_{train}(x) = w(x) \cdot P^*(x), \quad w(x) = \frac{\text{prob. de que } x \text{ esté en el corpus}}{P^*(x)}$$

Si $w(x) \not\equiv 1$ (lo que siempre ocurre), el estimador es **sesgado**.

El sesgo inducido en cualquier estadístico $T$ estimado desde $P_{train}$ es:
$$b(T) = E_{P_{train}}[T] - E_{P^*}[T] = E_{P^*}[(w(x)-1) \cdot T(x)]$$

In [ ]:
# Simulación: distribución verdadera P* vs corpus sesgado P_train
# Imaginamos 5 "regiones lingüísticas" con proporciones reales vs proporciones en corpus

regiones = ['Asia\nOriental', 'Asia\nMeridional', 'África\nSubsah.', 'Am. Latina\n+ Caribe', 'Europa +\nN. América']

# Proporción real de hablantes (aprox. a datos reales)
prop_real      = np.array([0.22, 0.20, 0.15, 0.14, 0.29])
# Proporción en corpus LLM típico (internet sesgado)
prop_corpus    = np.array([0.10, 0.05, 0.03, 0.07, 0.75])

# Factor de sobremuestreo w(x) por región
w = prop_corpus / prop_real

fig, axes = plt.subplots(1, 3, figsize=(14, 4))

colores = ['#e63946', '#457b9d', '#2a9d8f', '#e9c46a', '#264653']
x_pos = np.arange(len(regiones))

axes[0].bar(x_pos, prop_real * 100, color=colores, alpha=0.85)
axes[0].set_xticks(x_pos); axes[0].set_xticklabels(regiones, fontsize=8)
axes[0].set_ylabel('% de hablantes')
axes[0].set_title('$P^*$: Distribución real\nde hablantes mundiales')

axes[1].bar(x_pos, prop_corpus * 100, color=colores, alpha=0.85)
axes[1].set_xticks(x_pos); axes[1].set_xticklabels(regiones, fontsize=8)
axes[1].set_ylabel('% del corpus')
axes[1].set_title('$P_{train}$: Distribución típica\nen corpus LLM')

bars = axes[2].bar(x_pos, w, color=colores, alpha=0.85)
axes[2].axhline(1, color='black', ls='--', lw=1.2, label='$w=1$ (sin sesgo)')
axes[2].set_xticks(x_pos); axes[2].set_xticklabels(regiones, fontsize=8)
axes[2].set_ylabel('Factor $w(x) = P_{train}/P^*$')
axes[2].set_title('Factor de sobre/sub-muestreo\npor región')
axes[2].legend(fontsize=9)
for bar, val in zip(bars, w):
    axes[2].text(bar.get_x() + bar.get_width()/2, val + 0.05, f'{val:.2f}x',
                 ha='center', fontsize=8, fontweight='bold')

plt.suptitle('Sesgo de selección en el corpus de entrenamiento', fontweight='bold')
plt.tight_layout()
plt.show()

print("El factor w mide cuántas veces más (o menos) representada está cada región en el corpus.")
print(f"Europa + N. América: {w[-1]:.2f}x → sobrerepresentada")
print(f"África Subsahariana: {w[2]:.2f}x → subrepresentada")

### 1.2 Model collapse: amplificación del sesgo en entrenamiento recursivo

Un fenómeno emergente (Shumailov et al., 2024): cuando un LLM genera datos que luego se usan para entrenar la siguiente versión del modelo, el sesgo original se **amplifica** iteración a iteración. Es el análogo al estimador sesgado cuya distribución se aleja del parámetro verdadero cuando se itera.

Formalmente: si la generación $t$ produce $P_t(x) = $ LLM$_t$, y la generación $t+1$ se entrena sobre $P_t$ en lugar de $P^*$, entonces:

$$E_{P_{t+1}}[T] - E_{P^*}[T] \approx \alpha \cdot (E_{P_t}[T] - E_{P^*}[T]), \quad \alpha > 1$$

El sesgo **crece geométricamente** si $\alpha > 1$ (baja diversidad en los datos generados).

In [ ]:
# Simulación del model collapse: distribución de una característica (ej. sentimiento)
# P* = Normal(0, 1). El modelo aprende una versión sesgada y la usa para generar nuevos datos.

n_generaciones = 10
n_muestras_por_gen = 5000
mu_true = 0.0
sigma_true = 1.0

# Sesgo inicial en el corpus: el modelo aprende de datos con media 0.5 (sesgo positivo)
mu_corpus_0 = 0.5
sigma_shrink = 0.85  # cada generación reduce la varianza (pérdida de diversidad)

medias_generaciones = [mu_true, mu_corpus_0]  # gen 0: real, gen 1: corpus inicial
sigmas_generaciones = [sigma_true, sigma_true]

mu_actual = mu_corpus_0
sigma_actual = sigma_true

for g in range(2, n_generaciones + 1):
    # Cada generación aprende de la anterior + algo de ruido
    datos_gen = rng.normal(mu_actual, sigma_actual, n_muestras_por_gen)
    mu_actual = datos_gen.mean() + rng.normal(0, 0.05)  # pequeño ruido de estimación
    sigma_actual = datos_gen.std() * sigma_shrink
    medias_generaciones.append(mu_actual)
    sigmas_generaciones.append(sigma_actual)

generaciones = list(range(len(medias_generaciones)))
sesgos = [m - mu_true for m in medias_generaciones]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(generaciones, sesgos, 'o-', color='tomato', lw=2, label='Sesgo $E[\hat{\\mu}] - \\mu^*$')
axes[0].axhline(0, color='gray', ls=':', lw=1)
axes[0].fill_between(generaciones, sesgos, 0, alpha=0.2, color='tomato')
axes[0].set_xlabel('Generación de modelo')
axes[0].set_ylabel('Sesgo acumulado')
axes[0].set_title('Amplificación del sesgo\npor entrenamiento recursivo')
axes[0].legend()

axes[1].plot(generaciones, sigmas_generaciones, 's-', color='steelblue', lw=2, label='$\\sigma$ de la distribución generada')
axes[1].axhline(sigma_true, color='gray', ls=':', lw=1, label=f'$\\sigma^*={sigma_true}$')
axes[1].set_xlabel('Generación de modelo')
axes[1].set_ylabel('Desviación estándar')
axes[1].set_title('Colapso de diversidad\n(pérdida de varianza)')
axes[1].legend()

plt.suptitle('Model Collapse: sesgo creciente + pérdida de diversidad (Shumailov et al., 2024)',
             fontweight='bold')
plt.tight_layout()
plt.show()

print(f"Sesgo inicial (gen 1): {sesgos[1]:.3f}")
print(f"Sesgo en gen {n_generaciones}: {sesgos[-1]:.3f}")
print(f"Factor de amplificación: {sesgos[-1]/sesgos[1]:.2f}x")

---
## Parte 2 – El LLM como estimador: sesgo-varianza y double descent

### 2.1 Recap: la Cota de Cramér-Rao y la función de pérdida

En el Cap. 6 mostramos que para cualquier estimador insesgado $\hat{\theta}$:
$$\text{ECM}(\hat{\theta}) = V(\hat{\theta}) \geq \frac{1}{n \cdot I(\theta)}$$

En ML/LLMs esta idea generaliza al **riesgo de predicción** $R(f)$:
$$R(f) = E_{(x,y) \sim P^*}[\ell(f(x), y)] = \underbrace{\text{Sesgo}^2(f)}_\text{infraajuste} + \underbrace{V(f)}_\text{sobreajuste} + \underbrace{\sigma^2_\epsilon}_\text{ruido irreducible}$$

### 2.2 Double Descent: el tradeoff clásico quiebra para modelos grandes

En modelos clásicos (polinomios, redes pequeñas): al aumentar la complejidad, el riesgo cae, llega a un mínimo y luego sube (sobreajuste). Eso es el **tradeoff sesgo-varianza clásico**.

En LLMs (modelos sobre-parametrizados, Belkin et al., 2019): el riesgo vuelve a bajar después de cruzar la interpolación. La **curva de doble descenso** invalida la intuición clásica.

In [ ]:
# Visualización del double descent y comparación con tradeoff clásico
complejidad = np.linspace(0.1, 10, 1000)

# Curva clásica sesgo-varianza
sesgo2_clasico = 2.0 * np.exp(-0.8 * complejidad)
varianza_clasica = 0.1 * complejidad**1.5
riesgo_clasico = sesgo2_clasico + varianza_clasica + 0.1

# Curva double descent: al cruzar n_params = n_datos hay un pico de interpolación
# luego el riesgo vuelve a bajar en el régimen sobre-parametrizado
n_interp = 5.5  # punto de interpolación (n_params ≈ n_datos)

def double_descent(c):
    # Régimen sub-parametrizado: U-clásica
    # Régimen sobre-parametrizado: decreciente
    clasico = 2.0 * np.exp(-0.9*c) + 0.08 * c**1.4 + 0.1
    # Spike en la interpolación + caída sobre-parametrizada
    spike = 1.5 * np.exp(-3*(c - n_interp)**2)  # pico gaussiano
    over = np.where(c > n_interp, 0.25 / (c - n_interp + 0.5) - 0.15, 0)
    return clasico + spike + np.maximum(over, 0)

riesgo_dd = double_descent(complejidad)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Izquierda: tradeoff clásico
axes[0].plot(complejidad, sesgo2_clasico, '--', color='tomato', lw=1.5, label='Sesgo²')
axes[0].plot(complejidad, varianza_clasica, '--', color='steelblue', lw=1.5, label='Varianza')
axes[0].plot(complejidad, riesgo_clasico, '-', color='black', lw=2.5, label='Riesgo total')
axes[0].axvline(complejidad[np.argmin(riesgo_clasico)], color='green', ls=':', lw=1.5,
                label='Complejidad óptima')
axes[0].set_xlabel('Complejidad del modelo')
axes[0].set_ylabel('Error / Riesgo')
axes[0].set_title('Tradeoff sesgo–varianza clásico\n(modelos pequeños / estadística clásica)')
axes[0].legend(fontsize=9)
axes[0].set_ylim(0, 2.5)

# Derecha: double descent
axes[1].plot(complejidad, riesgo_dd, '-', color='darkorchid', lw=2.5, label='Riesgo (double descent)')
axes[1].axvline(n_interp, color='gray', ls='--', lw=1.5, label='Umbral de interpolación\n($n_{params} \\approx n_{datos}$)')
axes[1].fill_betweenx([0, 3.5], 0, n_interp, alpha=0.07, color='tomato', label='Régimen clásico')
axes[1].fill_betweenx([0, 3.5], n_interp, 10, alpha=0.07, color='steelblue', label='Sobre-param. (LLMs)')
axes[1].set_xlabel('Complejidad / nº parámetros')
axes[1].set_ylabel('Error / Riesgo')
axes[1].set_title('Double Descent\n(LLMs y modelos sobre-parametrizados)')
axes[1].legend(fontsize=8)
axes[1].set_ylim(0, 3.5)

plt.suptitle('Del ECM clásico al double descent: cómo cambia el tradeoff sesgo–varianza en LLMs',
             fontweight='bold')
plt.tight_layout()
plt.show()

### 2.3 LLMs como estimadores de parámetros latentes (Prediction-Powered Inference)

Un uso concreto: usar un LLM como **anotador automático** para estimar una proporción $\theta$ (por ejemplo, fracción de texto con sesgo político). El LLM produce anotaciones $\hat{Y}_i$ con error sistemático. Los estimadores derivados son sesgados.

**Marco formal (Angelopoulos et al., 2023 — Prediction-Powered Inference):**

- Dato real: $Y_i \in \{0, 1\}$, $i = 1, \ldots, n$ (etiquetas de experto)
- Predicción LLM: $\hat{Y}_i$ con sesgo $b = E[\hat{Y}_i] - E[Y_i] \neq 0$
- Estimador naive: $\hat{\theta}_{LLM} = \frac{1}{N}\sum_{i=1}^N \hat{Y}_i$ — **sesgado**
- Corrección (PPI): $\hat{\theta}_{PPI} = \hat{\theta}_{LLM} - \hat{b}$, donde $\hat{b}$ se estima con el subconjunto anotado manualmente

In [ ]:
# Demostración de sesgo en estimación via LLM-anotador y corrección PPI
theta_true_ppi = 0.35   # fracción real de documentos con sesgo político
N_total = 10_000        # corpus grande (anotado por LLM)
n_gold  = 200           # submuestra con anotación humana
N_sim_ppi = 5000

# El LLM sobreestima: tiene sensibilidad 0.90 pero especificidad 0.75
# (muchos falsos positivos -> sobreestima theta)
sens = 0.90   # P(LLM=1 | Y=1)
spec = 0.75   # P(LLM=0 | Y=0)  -> P(LLM=1 | Y=0) = 0.25

# Sesgo teórico:
# E[Y_hat] = sens * theta + (1-spec) * (1-theta)
E_Yhat = sens * theta_true_ppi + (1 - spec) * (1 - theta_true_ppi)
b_teorico = E_Yhat - theta_true_ppi
print(f"Sesgo teórico del estimador LLM: E[Ŷ] - θ = {b_teorico:.4f}")
print(f"E[Ŷ] = {E_Yhat:.4f} vs θ verdadero = {theta_true_ppi}")

theta_naive_list = []
theta_ppi_list   = []

for _ in range(N_sim_ppi):
    # Corpus completo
    Y_true   = (rng.random(N_total) < theta_true_ppi).astype(float)
    # Predicción LLM
    Y_hat = np.where(Y_true == 1,
                     (rng.random(N_total) < sens).astype(float),
                     (rng.random(N_total) < (1 - spec)).astype(float))
    theta_naive = Y_hat.mean()
    
    # Submuestra gold: estimar el sesgo
    idx_gold = rng.choice(N_total, n_gold, replace=False)
    b_hat = Y_hat[idx_gold].mean() - Y_true[idx_gold].mean()
    theta_ppi = theta_naive - b_hat
    
    theta_naive_list.append(theta_naive)
    theta_ppi_list.append(theta_ppi)

theta_naive_arr = np.array(theta_naive_list)
theta_ppi_arr   = np.array(theta_ppi_list)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

bins = np.linspace(0.2, 0.55, 60)
axes[0].hist(theta_naive_arr, bins=bins, density=True, alpha=0.7, color='tomato', label='$\\hat{\\theta}_{LLM}$ (sesgado)')
axes[0].hist(theta_ppi_arr,   bins=bins, density=True, alpha=0.7, color='seagreen', label='$\\hat{\\theta}_{PPI}$ (corregido)')
axes[0].axvline(theta_true_ppi, color='black', lw=2.5, ls='--', label=f'$\\theta={theta_true_ppi}$')
axes[0].set_xlabel(r'Estimación de $\theta$')
axes[0].set_ylabel('Densidad')
axes[0].set_title('Estimador LLM vs corregido PPI')
axes[0].legend(fontsize=9)

estimadores_ppi = {'LLM naive': theta_naive_arr, 'PPI corregido': theta_ppi_arr}
colores_ppi = ['tomato', 'seagreen']
for ax_i, (nombre, arr), color in zip([0, 1], estimadores_ppi.items(), colores_ppi):
    pass

# Panel derecho: tabla comparativa
axes[1].axis('off')
tabla_data = [
    ['Estimador', 'E[θ̂]', 'Sesgo', 'Var(θ̂)', 'ECM'],
    ['LLM naive',
     f'{theta_naive_arr.mean():.4f}',
     f'{theta_naive_arr.mean() - theta_true_ppi:.4f}',
     f'{theta_naive_arr.var():.6f}',
     f'{((theta_naive_arr - theta_true_ppi)**2).mean():.6f}'],
    ['PPI corregido',
     f'{theta_ppi_arr.mean():.4f}',
     f'{theta_ppi_arr.mean() - theta_true_ppi:.4f}',
     f'{theta_ppi_arr.var():.6f}',
     f'{((theta_ppi_arr - theta_true_ppi)**2).mean():.6f}']
]
tabla = axes[1].table(cellText=tabla_data[1:], colLabels=tabla_data[0],
                      loc='center', cellLoc='center')
tabla.auto_set_font_size(False)
tabla.set_fontsize(10)
tabla.scale(1.4, 2.0)
# Colorear filas
for j in range(5):
    tabla[(1, j)].set_facecolor('#ffd6d6')
    tabla[(2, j)].set_facecolor('#d6f5e3')
axes[1].set_title('Descomposición ECM: sesgo vs varianza', pad=60)

plt.suptitle('LLMs como anotadores: sesgo en estimación de parámetros (marco PPI)',
             fontweight='bold')
plt.tight_layout()
plt.show()

---
## Parte 3 – Sesgo cultural y geográfico

### 3.1 El problema: $E[\hat{f}(x)] \neq f^*(x)$ de forma diferencial por subpoblación

Traducido al lenguaje del Cap. 6: el estimador $\hat{f}$ tiene sesgo **diferencial** según el grupo. Para la subpoblación $g$:
$$b_g(\hat{f}) = E_{x \sim P_g}[\hat{f}(x)] - E_{x \sim P_g}[f^*(x)]$$

Si $b_g \neq b_{g'}$ para grupos $g \neq g'$, el modelo es **diferencialmente sesgado**.

**Evidencia empírica:** estudios usando el *World Values Survey* muestran que los LLMs se alinean consistentemente con valores occidentales (Europa + Norteamérica) incluso cuando se les pregunta en nombre de otras culturas (Tao et al., 2023; PNAS Nexus 2024). Un estudio con 107 países encontró que los outputs de 5 LLMs populares son significativamente más cercanos a los valores de países anglosajones que a los de los países evaluados.

In [ ]:
# Simulación de sesgo cultural diferencial
# Escenario: un LLM responde preguntas de valores en escala 1-10.
# La respuesta "verdadera" (según encuestas reales) varía por región cultural.
# El LLM tiene una distribución de respuestas centrada en los valores occidentales.

regiones_cult = [
    'Europa Occ.\n+ N. América',
    'Europa del Este\n+ Rusia',
    'Confuciana\n(China, Japón, Corea)',
    'Latinoamérica',
    'África\nSubsahariana',
    'Islam\n(Medio Oriente)'
]

# Valores medios reales (en dimensión "Valores de autoexpresión" del IVS, escala 0-1)
# Basado aproximadamente en el mapa Inglehart-Welzel
mu_real_cult   = np.array([0.75, 0.45, 0.55, 0.50, 0.35, 0.30])
sigma_real     = np.array([0.10, 0.12, 0.11, 0.12, 0.13, 0.12])

# El LLM "ancla" sus respuestas en el valor occidental (0.75) con algo de adaptación
# ancla = 0.90 * 0.75 + 0.10 * mu_real  (adaptación parcial al contexto)
alpha_ancla = 0.80
mu_llm = alpha_ancla * 0.75 + (1 - alpha_ancla) * mu_real_cult

sesgos_cult = mu_llm - mu_real_cult

N_cult_sim = 2000
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

colores_cult = ['#264653', '#2a9d8f', '#e9c46a', '#f4a261', '#e76f51', '#457b9d']
x_c = np.arange(len(regiones_cult))

bars_real = axes[0].bar(x_c - 0.2, mu_real_cult, 0.38, color=colores_cult,
                        alpha=0.85, label='Valor real (encuesta)')
bars_llm  = axes[0].bar(x_c + 0.2, mu_llm, 0.38, color=colores_cult,
                        alpha=0.4, hatch='//', label='Output LLM')
axes[0].set_xticks(x_c); axes[0].set_xticklabels(regiones_cult, fontsize=8)
axes[0].set_ylabel('Valor en dimensión IVS')
axes[0].set_title('Valores reales (encuesta) vs outputs LLM\npor región cultural')
axes[0].legend()

barras_sesgo = axes[1].bar(x_c, sesgos_cult, color=colores_cult, alpha=0.85)
axes[1].axhline(0, color='black', lw=1.2, ls='--')
axes[1].set_xticks(x_c); axes[1].set_xticklabels(regiones_cult, fontsize=8)
axes[1].set_ylabel('Sesgo diferencial $b_g = E[\hat{f}(x)] - f^*(x)$')
axes[1].set_title('Sesgo diferencial por región cultural\n(positivo = LLM sobrevalora autoexpresión occidental)')
for bar, val in zip(barras_sesgo, sesgos_cult):
    axes[1].text(bar.get_x() + bar.get_width()/2,
                 val + 0.005 if val >= 0 else val - 0.015,
                 f'{val:+.3f}', ha='center', fontsize=9, fontweight='bold')

plt.suptitle('Sesgo cultural diferencial en LLMs (inspirado en Tao et al. 2023 / PNAS Nexus 2024)',
             fontweight='bold')
plt.tight_layout()
plt.show()

print("\nConexión con Cap. 6:")
print(f"  Sesgo medio global:  {sesgos_cult.mean():+.4f}")
print(f"  Sesgo en Europa/NA:  {sesgos_cult[0]:+.4f}  (casi sin sesgo → modelo 'insesgado' para esta región)")
print(f"  Sesgo en Medio Este: {sesgos_cult[-1]:+.4f}  (fuertemente sesgado hacia arriba)")
print(f"  Varianza del sesgo:  {sesgos_cult.var():.6f}  (heterogeneidad del sesgo entre grupos)")

### 3.2 Detección de sesgo geográfico: disparidad en distribuciones de salida

Una métrica natural de auditoría es la **divergencia de Kullback-Leibler (KL)** entre la distribución de outputs del LLM y la distribución real para cada subpoblación:
$$D_{KL}(P_{real,g} \| P_{LLM,g}) = E_{P_{real,g}}\left[\log\frac{P_{real,g}(x)}{P_{LLM,g}(x)}\right]$$

Si $D_{KL}$ es sistemáticamente mayor para ciertas regiones, hay sesgo diferencial medible.

In [ ]:
# Cálculo de divergencia KL por región cultural
from scipy.special import kl_div

N_kl = 3000
kl_divergencias = []

for i, (mu_r, mu_l, sig) in enumerate(zip(mu_real_cult, mu_llm, sigma_real)):
    # Muestras de la distribución real y del LLM
    real_samples = rng.normal(mu_r, sig, N_kl)
    llm_samples  = rng.normal(mu_l, sig * 0.85, N_kl)  # LLM también reduce varianza
    
    # KL estimada via estimador de densidad en grilla
    grilla = np.linspace(0, 1, 200)
    kde_real = stats.gaussian_kde(np.clip(real_samples, 0.01, 0.99))
    kde_llm  = stats.gaussian_kde(np.clip(llm_samples,  0.01, 0.99))
    
    p = kde_real(grilla) + 1e-10
    q = kde_llm(grilla)  + 1e-10
    p /= p.sum(); q /= q.sum()
    
    kl = np.sum(p * np.log(p / q)) * (grilla[1] - grilla[0]) * len(grilla)
    kl_divergencias.append(kl)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

barras_kl = axes[0].bar(x_c, kl_divergencias, color=colores_cult, alpha=0.85)
axes[0].set_xticks(x_c); axes[0].set_xticklabels(regiones_cult, fontsize=8)
axes[0].set_ylabel(r'$D_{KL}(P_{real} \| P_{LLM})$')
axes[0].set_title('Divergencia KL por región cultural\n(mayor = LLM más alejado de la realidad local)')
for bar, val in zip(barras_kl, kl_divergencias):
    axes[0].text(bar.get_x() + bar.get_width()/2, val + 0.001,
                 f'{val:.3f}', ha='center', fontsize=9)

# Densidades superpuestas para el caso más extremo
idx_max = np.argmax(kl_divergencias)
mu_r_ex, mu_l_ex, sig_ex = mu_real_cult[idx_max], mu_llm[idx_max], sigma_real[idx_max]
grilla_ex = np.linspace(0, 1, 300)
kde_r_ex = stats.gaussian_kde(np.clip(rng.normal(mu_r_ex, sig_ex, 5000), 0.01, 0.99))
kde_l_ex = stats.gaussian_kde(np.clip(rng.normal(mu_l_ex, sig_ex*0.85, 5000), 0.01, 0.99))

axes[1].plot(grilla_ex, kde_r_ex(grilla_ex), color='steelblue', lw=2.5, label=f'Distribución real ({regiones_cult[idx_max].split(chr(10))[0]})')
axes[1].plot(grilla_ex, kde_l_ex(grilla_ex), color='tomato', lw=2.5, ls='--', label='Output LLM')
axes[1].fill_between(grilla_ex, kde_r_ex(grilla_ex), kde_l_ex(grilla_ex),
                     alpha=0.15, color='purple', label='Área de diferencia')
axes[1].set_xlabel('Valor en dimensión cultural')
axes[1].set_ylabel('Densidad')
axes[1].set_title(f'Región con mayor sesgo: {regiones_cult[idx_max].replace(chr(10), " ")}\n$D_{{KL}}={kl_divergencias[idx_max]:.3f}$')
axes[1].legend(fontsize=9)

plt.suptitle('Auditoría de sesgo cultural via Divergencia KL', fontweight='bold')
plt.tight_layout()
plt.show()

---
## Parte 4 – Sesgo en outputs: fairness, representación y estereotipos

### 4.1 Taxonomía del sesgo en outputs

Gallegos et al. (2024, *Computational Linguistics*) distinguen tres capas:

| Capa | Descripción | Analogía estadística |
|------|-------------|---------------------|
| **Sesgo de representación** | Subgrupos subrepresentados en los outputs | Muestra no representativa → estimador sesgado |
| **Sesgo de estereotipado** | Asociaciones entre grupo y atributo | Función $\hat{f}$ con sesgo sistemático por grupo |
| **Sesgo de daño** | Outputs dañinos disparatadamente hacia ciertos grupos | ECM diferencial entre grupos |

**Métrica clave:** Paridad demográfica. Para atributo $A$ (género, raza, etc.) y predicción $\hat{Y}$:
$$\Delta_{DP} = |P(\hat{Y}=1 | A=0) - P(\hat{Y}=1 | A=1)|$$
Si $\Delta_{DP} = 0$, hay paridad demográfica (fairness en el sentido estadístico más básico).

In [ ]:
# Demostración de sesgo en outputs: asociaciones estereotipadas
# Escenario: un LLM asigna scores de "adecuación para carrera" para distintos perfiles
# La única diferencia entre perfiles es el nombre (indicador de género/etnia)

np.random.seed(123)

n_por_grupo = 500

# Grupos y sus parámetros de distribución de scores (simulados, inspirados en estudios de audit)
grupos = {
    'Emily (F, blanca)':       {'mu': 7.1, 'sigma': 0.9},
    'Aisha (F, afrodesc.)':    {'mu': 6.2, 'sigma': 1.0},
    'Brad (M, blanco)':        {'mu': 7.5, 'sigma': 0.9},
    'Jamal (M, afrodesc.)':    {'mu': 6.3, 'sigma': 1.0},
    'María (F, latina)':       {'mu': 6.4, 'sigma': 1.0},
    'Carlos (M, latino)':      {'mu': 6.8, 'sigma': 0.9},
}

# Score "justo" que debería tener cada grupo si no hubiera sesgo
mu_justo = 7.0

scores = {}
for nombre, params in grupos.items():
    scores[nombre] = rng.normal(params['mu'], params['sigma'], n_por_grupo)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

colores_grupos = ['#e63946', '#457b9d', '#2a9d8f', '#e9c46a', '#f4a261', '#264653']
nombres = list(scores.keys())

# Boxplot
data_bp = [scores[n] for n in nombres]
bp = axes[0].boxplot(data_bp, patch_artist=True, notch=False)
for patch, color in zip(bp['boxes'], colores_grupos):
    patch.set_facecolor(color)
    patch.set_alpha(0.75)
axes[0].axhline(mu_justo, color='black', ls='--', lw=1.5, label='Score "justo" esperado')
axes[0].set_xticks(range(1, len(nombres)+1))
axes[0].set_xticklabels([n.split('(')[0].strip() for n in nombres], fontsize=9)
axes[0].set_ylabel('Score de adecuación (1–10)')
axes[0].set_title('Distribución de scores por grupo\n(mismo CV, distinto nombre)')
axes[0].legend()

# Sesgo diferencial y paridad demográfica
medias = [scores[n].mean() for n in nombres]
sesgos_output = [m - mu_justo for m in medias]

colores_sesgo = ['tomato' if s < 0 else 'seagreen' for s in sesgos_output]
barras_out = axes[1].bar(range(len(nombres)), sesgos_output, color=colores_sesgo, alpha=0.8)
axes[1].axhline(0, color='black', lw=1.2, ls='--')
axes[1].set_xticks(range(len(nombres)))
axes[1].set_xticklabels([n.split('(')[0].strip() for n in nombres], fontsize=9)
axes[1].set_ylabel('Sesgo $b_g = E[\\hat{Y}|A=g] - \\mu_{justo}$')
axes[1].set_title('Sesgo diferencial por grupo en los outputs')
for bar, val in zip(barras_out, sesgos_output):
    axes[1].text(bar.get_x() + bar.get_width()/2,
                 val + 0.02 if val >= 0 else val - 0.08,
                 f'{val:+.2f}', ha='center', fontsize=9, fontweight='bold')

plt.suptitle('Sesgo en outputs: disparidad de scores por nombre/grupo demográfico',
             fontweight='bold')
plt.tight_layout()
plt.show()

# Paridad demográfica: comparación F vs M
grupos_F = ['Emily (F, blanca)', 'Aisha (F, afrodesc.)', 'María (F, latina)']
grupos_M = ['Brad (M, blanco)',  'Jamal (M, afrodesc.)', 'Carlos (M, latino)']

umbral = 7.0
P_F = np.mean([np.mean(scores[g] >= umbral) for g in grupos_F])
P_M = np.mean([np.mean(scores[g] >= umbral) for g in grupos_M])

print(f"\nParidad demográfica (score ≥ {umbral}):")
print(f"  P(Ŷ≥{umbral} | A=Femenino): {P_F:.3f}")
print(f"  P(Ŷ≥{umbral} | A=Masculino): {P_M:.3f}")
print(f"  Δ_DP = {abs(P_F - P_M):.3f}  (0 = paridad perfecta)")

### 4.2 Las métricas de fairness son inconsistentes entre sí

Un resultado importante: es **matemáticamente imposible** satisfacer simultáneamente las principales métricas de fairness cuando las tasas base difieren entre grupos (Chouldechova, 2017; Kleinberg et al., 2016). Esta es una extensión directa de la imposibilidad de que un estimador sesgado sea simultáneamente eficiente en todos los sentidos.

Las tres métricas más comunes:
- **Paridad demográfica:** $P(\hat{Y}=1|A=0) = P(\hat{Y}=1|A=1)$
- **Igualdad de oportunidades:** $P(\hat{Y}=1|Y=1, A=0) = P(\hat{Y}=1|Y=1, A=1)$ (igual sensibilidad)
- **Calibración:** $P(Y=1|\hat{Y}=s, A=0) = P(Y=1|\hat{Y}=s, A=1)$ (igual precisión por score)

In [ ]:
# Demostración del teorema de imposibilidad de fairness
# Dos grupos con tasas base distintas: P(Y=1|A=0) ≠ P(Y=1|A=1)

N_fair = 10_000

# Tasas base: grupo 0 tiene 30% positivos, grupo 1 tiene 50% positivos
p0_base = 0.30   # prevalencia en grupo 0
p1_base = 0.50   # prevalencia en grupo 1

# Un clasificador calibrado perfecto: score ≈ P(Y=1|x)
Y0 = (rng.random(N_fair) < p0_base).astype(int)
Y1 = (rng.random(N_fair) < p1_base).astype(int)

# Scores del clasificador: señal real + ruido
score0 = np.clip(Y0 * rng.normal(0.7, 0.15, N_fair) + (1-Y0) * rng.normal(0.3, 0.15, N_fair), 0, 1)
score1 = np.clip(Y1 * rng.normal(0.7, 0.15, N_fair) + (1-Y1) * rng.normal(0.3, 0.15, N_fair), 0, 1)

umbrales = np.linspace(0.1, 0.9, 100)

# Para cada umbral, calcula las tres métricas
parity, eq_opp, prec_gap = [], [], []

for t in umbrales:
    Yhat0 = (score0 >= t).astype(int)
    Yhat1 = (score1 >= t).astype(int)
    
    # Paridad demográfica
    dp0 = Yhat0.mean()
    dp1 = Yhat1.mean()
    parity.append(abs(dp0 - dp1))
    
    # Igualdad de oportunidades (sensibilidad en Y=1)
    tpr0 = Yhat0[Y0==1].mean() if Y0.sum() > 0 else 0
    tpr1 = Yhat1[Y1==1].mean() if Y1.sum() > 0 else 0
    eq_opp.append(abs(tpr0 - tpr1))
    
    # Brecha de precisión (PPV / valor predictivo positivo)
    ppv0 = Y0[Yhat0==1].mean() if Yhat0.sum() > 0 else 0
    ppv1 = Y1[Yhat1==1].mean() if Yhat1.sum() > 0 else 0
    prec_gap.append(abs(ppv0 - ppv1))

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].plot(umbrales, parity,   color='tomato',    lw=2, label='Δ Paridad demográfica')
axes[0].plot(umbrales, eq_opp,   color='steelblue', lw=2, label='Δ Igualdad de oportunidades')
axes[0].plot(umbrales, prec_gap, color='seagreen',  lw=2, label='Δ Precisión (PPV)')
axes[0].axhline(0, color='gray', ls=':', lw=1)
axes[0].set_xlabel('Umbral de decisión')
axes[0].set_ylabel('|Diferencia entre grupos|')
axes[0].set_title(f'Métricas de fairness vs umbral\n(tasas base: $p_0={p0_base}$, $p_1={p1_base}$)')
axes[0].legend(fontsize=9)

# Punto donde cada métrica se minimiza
t_parity  = umbrales[np.argmin(parity)]
t_eq_opp  = umbrales[np.argmin(eq_opp)]
t_prec    = umbrales[np.argmin(prec_gap)]

axes[1].axis('off')
texto_imp = [
    "Teorema de imposibilidad (Chouldechova 2017):",
    "",
    "Si las tasas base difieren entre grupos,",
    "NO existe un clasificador que satisfaga",
    "simultáneamente:",
    "",
    "  • Paridad demográfica",
    "  • Igualdad de oportunidades",
    "  • Calibración perfecta",
    "",
    f"Umbral óptimo para paridad:  t* = {t_parity:.2f}",
    f"Umbral óptimo para eq. oport.: t* = {t_eq_opp:.2f}",
    f"Umbral óptimo para precisión: t* = {t_prec:.2f}",
    "",
    "→ Análogo al Cap. 6: no existe un estimador",
    "  que minimice ECM, Sesgo Y Varianza",
    "  simultáneamente para todos los grupos."
]
for i, linea in enumerate(texto_imp):
    peso = 'bold' if i == 0 or '→' in linea else 'normal'
    axes[1].text(0.05, 0.97 - i * 0.058, linea, transform=axes[1].transAxes,
                 fontsize=10, fontweight=peso, va='top',
                 color='darkred' if '→' in linea else 'black')

plt.suptitle('Imposibilidad de fairness simultánea (con tasas base distintas)', fontweight='bold')
plt.tight_layout()
plt.show()

---
## Parte 5 – Síntesis: mapa conceptual de los cuatro sesgos

### 5.1 Todos los sesgos son instancias del sesgo estadístico del Cap. 6

| Tipo de sesgo | Formulación estadística | ¿Qué se estima? | Herramienta de detección |
|---|---|---|---|
| **Datos de entrenamiento** | $E_{P_{train}}[T] \neq E_{P^*}[T]$ | Distribución subyacente $P^*$ | Análisis de corpus, importance weighting |
| **Estimación (model collapse)** | $b_t \to \infty$ iterativamente | Parámetros del modelo | Diversidad de generaciones |
| **Cultural/geográfico** | $b_g \neq b_{g'}$ entre grupos | Valores/hechos por subpoblación | Benchmarks culturales, KL divergence |
| **Outputs (fairness)** | $\Delta_{DP}, \Delta_{EO} \neq 0$ | Tasa de decisión justa | Auditorías, métricas de paridad |

### 5.2 El triángulo sesgo–varianza–ECM en LLMs

In [ ]:
# Visualización síntesis: los 4 tipos de sesgo en un solo gráfico
fig, ax = plt.subplots(figsize=(12, 7))
ax.set_xlim(0, 10); ax.set_ylim(0, 8)
ax.axis('off')

# Nodo central
centro = plt.Circle((5, 4), 0.7, color='#1d3557', zorder=3)
ax.add_patch(centro)
ax.text(5, 4, 'LLM\n$\\hat{f}$', ha='center', va='center', color='white',
        fontsize=11, fontweight='bold', zorder=4)

# Nodos de los 4 tipos de sesgo
nodos = [
    (2.0, 6.5, '#e63946', '1. Sesgo en\ndatos\n$P_{train} \\neq P^*$'),
    (8.0, 6.5, '#457b9d', '2. Sesgo en\nestimación\n(double descent)'),
    (2.0, 1.5, '#2a9d8f', '3. Sesgo\ncultural/geogr.\n$b_g \\neq b_{g\'}$'),
    (8.0, 1.5, '#e9c46a', '4. Sesgo en\noutputs\n(fairness)'),
]

for (x, y, color, texto) in nodos:
    rect = mpatches.FancyBboxPatch((x - 1.1, y - 0.65), 2.2, 1.3,
                                    boxstyle='round,pad=0.1',
                                    facecolor=color, alpha=0.85, zorder=2)
    ax.add_patch(rect)
    ax.text(x, y, texto, ha='center', va='center', fontsize=9,
            color='white' if color != '#e9c46a' else 'black',
            fontweight='bold', zorder=3)
    # Flecha al centro
    dx = 5 - x; dy = 4 - y
    norm = np.sqrt(dx**2 + dy**2)
    ax.annotate('', xy=(5 - 0.75*dx/norm, 4 - 0.75*dy/norm),
                xytext=(x + 1.1*dx/norm, y + 0.65*dy/norm),
                arrowprops=dict(arrowstyle='->', color='gray', lw=1.5))

# Anotaciones con conexión al Cap. 6
anotaciones = [
    (0.05, 0.97, 'Cap. 6 → LLMs', '#1d3557', 14),
    (0.05, 0.90, '$b(\\hat{\\theta}) = E(\\hat{\\theta}) - \\theta$', '#1d3557', 11),
    (0.05, 0.83, '$\\text{ECM} = V(\\hat{\\theta}) + b^2$', '#1d3557', 11),
    (0.05, 0.76, 'CCR: $V(\\hat{\\theta}) \\geq 1/(nI(\\theta))$', '#1d3557', 11),
]
for (relx, rely, txt, col, fs) in anotaciones:
    ax.text(relx * 10, rely * 8, txt, fontsize=fs, color=col,
            fontweight='bold' if fs == 14 else 'normal')

ax.set_title('Mapa conceptual: cuatro tipos de sesgo en LLMs\n'
             'bajo el marco del Cap. 6 (Estimación Puntual)',
             fontsize=13, fontweight='bold', pad=15)
plt.tight_layout()
plt.show()

In [ ]:
# Tabla-resumen: estrategias de mitigación
print("="*80)
print("RESUMEN: Tipos de sesgo en LLMs y estrategias de mitigación")
print("="*80)

filas = [
    ("1. Datos de entrenamiento",
     "Corpus sesgado hacia inglés / Global North",
     "Importance weighting, curación de datos",
     "Alta: afecta a todos los usuarios"),
    
    ("2. Model collapse",
     "Entrenamiento recursivo amplifica sesgo y reduce diversidad",
     "Mantener datos humanos en cada ciclo",
     "Alta: riesgo sistémico a largo plazo"),
    
    ("3. Cultural / Geográfico",
     "Valores occidentales sobre-representados en outputs",
     "Benchmarks culturales, fine-tuning localizado",
     "Alta: impacto en ~80% de la población mundial"),
    
    ("4. Outputs (fairness)",
     "Decisiones disparatadas por género, etnia, origen",
     "Auditorías, re-calibración, RLHF con restricciones",
     "Alta en aplicaciones de alto impacto"),
]

print(f"\n{'Tipo':<28} {'Descripción':<40} {'Mitigación':<35} {'Impacto'}")
print('-' * 130)
for fila in filas:
    print(f"{fila[0]:<28} {fila[1]:<40} {fila[2]:<35} {fila[3]}")

print("\n" + "="*80)
print("CONEXIÓN CON CAP. 6:")
print("  - Todos los sesgos son instancias de b(θ̂) = E(θ̂) - θ con θ latente.")
print("  - La descomposición ECM = Varianza + Sesgo² aparece en cada capa.")
print("  - La imposibilidad de fairness ≅ imposibilidad de minimizar ECM")
print("    simultáneamente en todas las subpoblaciones.")
print("  - La CCR establece un piso: no se puede reducir varianza + sesgo a cero")
print("    con datos limitados y distribución no representativa.")
print("="*80)